# DogFish + InceptionV3 Influence Explanation (PyTorch)

This notebook reproduces the core DogFish/Inception influence workflow from the original TensorFlow repo:

```text
dog/fish local images -> frozen ImageNet InceptionV3 features -> binary logistic top model -> influence ranking
```

It explains a selected test image prediction by showing the most helpful and harmful training images. It does **not** implement the full poisoning attack loop.

In [ ]:
import gc
import math
import os
import time
import tracemalloc
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# Make relative paths work whether the notebook is opened from repo root or test/.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "dogfish_inception_explain_pytorch.ipynb").exists():
    candidate = NOTEBOOK_DIR / "influence-release" / "test"
    if candidate.exists():
        os.chdir(candidate)
        NOTEBOOK_DIR = Path.cwd()

REPO_ROOT = NOTEBOOK_DIR.parent
DATA_ROOT = REPO_ROOT / "data"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

CONFIG_NAME = "demo"  # Change to "repo" for the larger original-style setup.

CONFIGS = {
    "demo": {
        "classes": ["dog", "fish"],
        "train_per_class": 50,
        "test_per_class": 20,
        "feature_batch_size": 16,
        "weight_decay": 1e-3,
        "lbfgs_max_iter": 50,
        "top_k": 5,
        "selected_method": "cg",
        "cg_max_iter": 50,
        "cg_tol": 1e-6,
        "lissa_batch_size": 32,
        "lissa_depth": 100,
        "lissa_scale": 25.0,
        "lissa_num_samples": 1,
        "sanity_checks": 2,
        "test_index": 0,
        "seed": 0,
    },
    "repo": {
        "classes": ["dog", "fish"],
        "train_per_class": 900,
        "test_per_class": 300,
        "feature_batch_size": 30,
        "weight_decay": 1e-3,
        "lbfgs_max_iter": 1000,
        "top_k": 10,
        "selected_method": "cg",
        "cg_max_iter": 100,
        "cg_tol": 1e-6,
        "lissa_batch_size": 100,
        "lissa_depth": 5000,
        "lissa_scale": 25.0,
        "lissa_num_samples": 1,
        "sanity_checks": 3,
        "test_index": 0,
        "seed": 0,
    },
}

cfg = CONFIGS[CONFIG_NAME]
torch.manual_seed(cfg["seed"])
np.random.seed(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"CONFIG_NAME = {CONFIG_NAME}")
print(f"device = {device}")
for key, value in cfg.items():
    print(f"{key}: {value}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 1. Load DogFish Image Paths

Expected local folder structure:

```text
influence-release/data/dog/
influence-release/data/fish/
```

The original repo builds DogFish from ImageNet animal synsets. This notebook assumes those images already exist locally.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_image_paths(class_name: str) -> List[Path]:
    class_dir = DATA_ROOT / class_name
    if not class_dir.exists():
        raise FileNotFoundError(
            f"Missing folder: {class_dir}\n"
            "Expected DogFish-style folders under influence-release/data/, e.g. data/dog and data/fish."
        )
    paths = [p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
    paths = sorted(paths, key=lambda p: p.name)
    if not paths:
        raise FileNotFoundError(f"No image files found in {class_dir}")
    return paths


def build_dogfish_split(cfg: Dict) -> Tuple[List[Path], Tensor, List[Path], Tensor]:
    train_paths, train_labels = [], []
    test_paths, test_labels = [], []

    for label, class_name in enumerate(cfg["classes"]):
        paths = list_image_paths(class_name)
        needed = cfg["train_per_class"] + cfg["test_per_class"]
        if len(paths) < needed:
            raise ValueError(
                f"Class '{class_name}' has {len(paths)} images, but config needs {needed}. "
                "Lower train_per_class/test_per_class or add more images."
            )

        train_part = paths[: cfg["train_per_class"]]
        test_part = paths[cfg["train_per_class"] : needed]
        train_paths.extend(train_part)
        train_labels.extend([label] * len(train_part))
        test_paths.extend(test_part)
        test_labels.extend([label] * len(test_part))

    rng = np.random.default_rng(cfg["seed"])
    train_perm = rng.permutation(len(train_paths))
    test_perm = rng.permutation(len(test_paths))

    train_paths = [train_paths[i] for i in train_perm]
    test_paths = [test_paths[i] for i in test_perm]
    train_labels = torch.tensor([train_labels[i] for i in train_perm], dtype=torch.float32)
    test_labels = torch.tensor([test_labels[i] for i in test_perm], dtype=torch.float32)
    return train_paths, train_labels, test_paths, test_labels


train_paths, train_y_cpu, test_paths, test_y_cpu = build_dogfish_split(cfg)
print(f"train images: {len(train_paths)}")
print(f"test images : {len(test_paths)}")
for class_name in cfg["classes"]:
    print(f"{class_name}: {len(list_image_paths(class_name))} local images")

In [ ]:
def show_image_grid(paths: List[Path], labels: Tensor, title: str, max_images: int = 8) -> None:
    count = min(len(paths), max_images)
    cols = min(count, 4)
    rows = math.ceil(count / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 3.2 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes[count:]:
        ax.axis("off")
    for ax, path, label in zip(axes, paths[:count], labels[:count]):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{path.name}\nlabel={cfg['classes'][int(label.item())]}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


show_image_grid(train_paths, train_y_cpu, "Training image samples")
show_image_grid(test_paths, test_y_cpu, "Test image samples")

## 2. Frozen InceptionV3 Feature Extraction

The original repo uses Keras/TensorFlow InceptionV3 without the top classifier and then trains a binary top model. Here we use torchvision's ImageNet-pretrained InceptionV3 and freeze it.

Feature saving code is included but commented out by default.

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths: List[Path], labels: Tensor, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        image = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(image), self.labels[idx], str(self.paths[idx])


def make_inception_feature_extractor(device: torch.device) -> Tuple[nn.Module, transforms.Compose]:
    weights = models.Inception_V3_Weights.IMAGENET1K_V1
    model = models.inception_v3(weights=weights)
    model.fc = nn.Identity()
    model.eval().to(device)
    for param in model.parameters():
        param.requires_grad_(False)

    # Close to the original repo: resize directly to 299x299 and use ImageNet normalization.
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return model, transform


def feature_cache_path(split: str) -> Path:
    return OUTPUT_DIR / f"dogfish_inception_{CONFIG_NAME}_{split}_features.npz"


def load_feature_cache(split: str):
    path = feature_cache_path(split)
    if not path.exists():
        return None
    cache = np.load(path, allow_pickle=True)
    print(f"Loaded cached {split} features from {path}")
    return (
        torch.tensor(cache["features"], dtype=torch.float32),
        torch.tensor(cache["labels"], dtype=torch.float32),
        [Path(p) for p in cache["paths"].tolist()],
    )


def extract_inception_features(paths: List[Path], labels: Tensor, split: str):
    cached = load_feature_cache(split)
    if cached is not None:
        return cached

    extractor, transform = make_inception_feature_extractor(device)
    dataset = ImagePathDataset(paths, labels, transform)
    loader = DataLoader(dataset, batch_size=cfg["feature_batch_size"], shuffle=False, num_workers=0)

    features, labels_out, paths_out = [], [], []
    with torch.no_grad():
        for images, batch_labels, batch_paths in loader:
            images = images.to(device)
            batch_features = extractor(images)
            if isinstance(batch_features, tuple):
                batch_features = batch_features[0]
            features.append(batch_features.detach().cpu())
            labels_out.append(batch_labels.detach().cpu())
            paths_out.extend([Path(p) for p in batch_paths])

    features = torch.cat(features, dim=0)
    labels_out = torch.cat(labels_out, dim=0).float()

    # OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # np.savez(
    #     feature_cache_path(split),
    #     features=features.numpy(),
    #     labels=labels_out.numpy(),
    #     paths=np.array([str(p) for p in paths_out], dtype=object),
    #     config_name=CONFIG_NAME,
    # )
    # print(f"Saved {split} feature cache to {feature_cache_path(split)}")

    return features, labels_out, paths_out


train_x_cpu, train_y_cpu, train_paths = extract_inception_features(train_paths, train_y_cpu, "train")
test_x_cpu, test_y_cpu, test_paths = extract_inception_features(test_paths, test_y_cpu, "test")

train_x = train_x_cpu.to(device)
train_y = train_y_cpu.to(device)
test_x = test_x_cpu.to(device)
test_y = test_y_cpu.to(device)

print("train features:", tuple(train_x.shape))
print("test features :", tuple(test_x.shape))

## 3. Train Binary Logistic Top Model

The original top model is binary logistic regression on frozen Inception features, trained with LBFGS. We match that with PyTorch `LBFGS` and no bias term.

In [ ]:
class BinaryLogisticTop(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.linear(x).squeeze(-1)


def top_objective(model: BinaryLogisticTop, x: Tensor, y: Tensor, weight_decay: float) -> Tensor:
    logits = model(x)
    data_loss = F.binary_cross_entropy_with_logits(logits, y, reduction="mean")
    reg_loss = 0.5 * weight_decay * model.linear.weight.reshape(-1).dot(model.linear.weight.reshape(-1))
    return data_loss + reg_loss


def train_top_model(x: Tensor, y: Tensor, cfg: Dict) -> BinaryLogisticTop:
    torch.manual_seed(cfg["seed"])
    model = BinaryLogisticTop(x.shape[1]).to(x.device)
    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=cfg["lbfgs_max_iter"],
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()
        loss = top_objective(model, x, y, cfg["weight_decay"])
        loss.backward()
        return loss

    optimizer.step(closure)
    return model


def evaluate_top_model(model: BinaryLogisticTop, x: Tensor, y: Tensor) -> Dict[str, float]:
    with torch.no_grad():
        logits = model(x)
        loss = F.binary_cross_entropy_with_logits(logits, y, reduction="mean").item()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        acc = (preds == y).float().mean().item()
        tp = int(((preds == 1) & (y == 1)).sum().item())
        tn = int(((preds == 0) & (y == 0)).sum().item())
        fp = int(((preds == 1) & (y == 0)).sum().item())
        fn = int(((preds == 0) & (y == 1)).sum().item())
    return {"loss": loss, "acc": acc, "tp": tp, "tn": tn, "fp": fp, "fn": fn}


top_model = train_top_model(train_x, train_y, cfg)
train_metrics = evaluate_top_model(top_model, train_x, train_y)
test_metrics = evaluate_top_model(top_model, test_x, test_y)
print("Train metrics:", train_metrics)
print("Test metrics :", test_metrics)
print("Confusion matrix on test: rows=true, cols=pred")
print(np.array([[test_metrics["tn"], test_metrics["fp"]], [test_metrics["fn"], test_metrics["tp"]]]))

In [ ]:
test_index = int(cfg["test_index"])
if test_index >= len(test_paths):
    raise ValueError(f"test_index={test_index} is out of range for {len(test_paths)} test images")

with torch.no_grad():
    test_logit = top_model(test_x[test_index : test_index + 1])
    test_prob = torch.sigmoid(test_logit).item()
    test_loss = F.binary_cross_entropy_with_logits(test_logit, test_y[test_index : test_index + 1], reduction="sum").item()

img = Image.open(test_paths[test_index]).convert("RGB")
plt.figure(figsize=(3.5, 3.5))
plt.imshow(img)
plt.axis("off")
plt.title(
    f"test idx={test_index}\n"
    f"true={cfg['classes'][int(test_y[test_index].item())]} | P(fish)={test_prob:.4f} | loss={test_loss:.4f}",
    fontsize=10,
)
plt.tight_layout()
plt.show()

## 4. Influence Helpers

These functions operate on the **top logistic model over cached Inception features**, matching the original repo's feature-level influence explanation.

In [ ]:
def tensor_memory_mb(*tensors: Tensor) -> float:
    return sum(t.numel() * t.element_size() for t in tensors if t is not None) / (1024 ** 2)


def current_rss_mb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    except Exception:
        return None


def per_example_weight_grads(model: BinaryLogisticTop, x: Tensor, y: Tensor) -> Tensor:
    grads = []
    weight = model.linear.weight
    for i in range(x.shape[0]):
        logit = model(x[i : i + 1])
        loss = F.binary_cross_entropy_with_logits(logit, y[i : i + 1], reduction="sum")
        grad = torch.autograd.grad(loss, weight, retain_graph=False, create_graph=False)[0]
        grads.append(grad.reshape(-1).detach())
    return torch.stack(grads, dim=0)


def logistic_hessian_autograd(model: BinaryLogisticTop, x: Tensor, y: Tensor, weight_decay: float, damping: float) -> Tensor:
    weight = model.linear.weight.detach().reshape(-1).requires_grad_(True)

    def objective(flat_weight: Tensor) -> Tensor:
        logits = x.matmul(flat_weight)
        data_loss = F.binary_cross_entropy_with_logits(logits, y, reduction="mean")
        reg_loss = 0.5 * weight_decay * flat_weight.dot(flat_weight)
        return data_loss + reg_loss

    hessian = torch.autograd.functional.hessian(objective, weight, vectorize=True)
    eye = torch.eye(hessian.shape[0], device=x.device, dtype=x.dtype)
    return hessian.detach() + damping * eye


def hessian_vector_product_autograd(
    model: BinaryLogisticTop,
    x: Tensor,
    y: Tensor,
    vector: Tensor,
    weight_decay: float,
    damping: float = 0.0,
) -> Tensor:
    flat_weight = model.linear.weight.detach().reshape(-1).requires_grad_(True)
    vector = vector.detach()
    logits = x.matmul(flat_weight)
    data_loss = F.binary_cross_entropy_with_logits(logits, y, reduction="mean")
    reg_loss = 0.5 * weight_decay * flat_weight.dot(flat_weight)
    loss = data_loss + reg_loss
    grad = torch.autograd.grad(loss, flat_weight, create_graph=True)[0]
    hvp = torch.autograd.grad(grad.dot(vector), flat_weight, retain_graph=False)[0]
    return hvp.detach() + damping * vector


def inverse_hvp_naive_hessian(model, x, y, vector, weight_decay, damping):
    hessian = logistic_hessian_autograd(model, x, y, weight_decay, damping)
    inverse_hvp = torch.linalg.solve(hessian, vector)
    residual = torch.linalg.norm(hessian.matmul(inverse_hvp) - vector).item()
    return inverse_hvp.detach(), {"tensor_est_mb": tensor_memory_mb(hessian, inverse_hvp), "iterations": 1, "residual": residual}


def inverse_hvp_cg(model, x, y, vector, weight_decay, damping, max_iter=100, tol=1e-6):
    solution = torch.zeros_like(vector)
    residual = vector.clone()
    direction = residual.clone()
    residual_sq = residual.dot(residual)
    initial_norm = torch.sqrt(residual_sq).clamp_min(1e-12)
    last_iter = 0

    for last_iter in range(1, max_iter + 1):
        h_direction = hessian_vector_product_autograd(model, x, y, direction, weight_decay, damping)
        denom = direction.dot(h_direction).clamp_min(1e-12)
        alpha = residual_sq / denom
        solution = solution + alpha * direction
        residual = residual - alpha * h_direction
        new_residual_sq = residual.dot(residual)
        if (torch.sqrt(new_residual_sq) / initial_norm).item() < tol:
            residual_sq = new_residual_sq
            break
        beta = new_residual_sq / residual_sq.clamp_min(1e-12)
        direction = residual + beta * direction
        residual_sq = new_residual_sq

    return solution.detach(), {
        "tensor_est_mb": tensor_memory_mb(solution, residual, direction),
        "iterations": last_iter,
        "residual": float(torch.sqrt(residual_sq).detach().cpu()),
    }


def inverse_hvp_lissa(model, x, y, vector, weight_decay, damping, batch_size=100, recursion_depth=500, scale=25.0, num_samples=1):
    n = x.shape[0]
    estimates = []
    for _ in range(num_samples):
        cur_estimate = vector.clone()
        for _ in range(recursion_depth):
            batch_indices = torch.randint(0, n, (min(batch_size, n),), device=x.device)
            hvp = hessian_vector_product_autograd(model, x[batch_indices], y[batch_indices], cur_estimate, weight_decay, damping=0.0)
            # Repo-style LiSSA update: v + (1 - damping) * s - H_batch s / scale.
            cur_estimate = vector + (1.0 - damping) * cur_estimate - hvp / scale
        estimates.append(cur_estimate / scale)

    inverse_hvp = torch.stack(estimates, dim=0).mean(dim=0)
    full_residual = hessian_vector_product_autograd(model, x, y, inverse_hvp, weight_decay, damping) - vector
    return inverse_hvp.detach(), {
        "tensor_est_mb": tensor_memory_mb(inverse_hvp, cur_estimate),
        "iterations": recursion_depth * num_samples,
        "residual": float(torch.linalg.norm(full_residual).detach().cpu()),
    }


def benchmark_inverse_hvp(method_name: str, fn):
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)

    rss_before = current_rss_mb()
    tracemalloc.start()
    start = time.perf_counter()
    inverse_hvp, info = fn()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    _, python_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after = current_rss_mb()

    row = {
        "method": method_name,
        "time_sec": elapsed,
        "python_peak_mb": python_peak / (1024 ** 2),
        "rss_delta_mb": None if rss_before is None or rss_after is None else rss_after - rss_before,
        "cuda_peak_mb": 0.0 if device.type != "cuda" else torch.cuda.max_memory_allocated(device) / (1024 ** 2),
    }
    row.update(info)
    return inverse_hvp, row


def print_benchmark_table(rows):
    print("\nInverse-HVP benchmark")
    print(" method          | time(s) | tensor_est(MB) | py_peak(MB) | rss_delta(MB) | cuda_peak(MB) | iters | residual")
    print("-----------------+---------+----------------+-------------+---------------+---------------+-------+----------")
    for row in rows:
        rss = "n/a" if row["rss_delta_mb"] is None else f"{row['rss_delta_mb']:.2f}"
        print(
            f" {row['method']:<15} | {row['time_sec']:>7.3f} | {row['tensor_est_mb']:>14.2f} | "
            f"{row['python_peak_mb']:>11.2f} | {rss:>13} | {row['cuda_peak_mb']:>13.2f} | "
            f"{row['iterations']:>5} | {row['residual']:.2e}"
        )


def upweight_effect(train_grad: Tensor, inverse_hvp: Tensor) -> Tensor:
    return -train_grad.matmul(inverse_hvp)


def leave_one_out_approx(upweight_scores: Tensor, num_train: int) -> Tensor:
    return -upweight_scores / num_train


def helpful_harmful_indices(loo_scores: Tensor, top_k: int) -> Tuple[Tensor, Tensor]:
    helpful = torch.argsort(loo_scores, descending=True)[:top_k]
    harmful = torch.argsort(loo_scores, descending=False)[:top_k]
    return helpful, harmful

## 5. Compute Influence Rankings

In [ ]:
test_feature = test_x[test_index : test_index + 1]
test_label = test_y[test_index : test_index + 1]

train_grads = per_example_weight_grads(top_model, train_x, train_y)
test_grad = per_example_weight_grads(top_model, test_feature, test_label).squeeze(0)

inverse_hvp_results = {}
benchmark_rows = []

inverse_hvp_results["naive_hessian"], row = benchmark_inverse_hvp(
    "naive_hessian",
    lambda: inverse_hvp_naive_hessian(top_model, train_x, train_y, test_grad, cfg["weight_decay"], damping=1e-4),
)
benchmark_rows.append(row)

inverse_hvp_results["cg"], row = benchmark_inverse_hvp(
    "cg",
    lambda: inverse_hvp_cg(
        top_model,
        train_x,
        train_y,
        test_grad,
        cfg["weight_decay"],
        damping=1e-4,
        max_iter=cfg["cg_max_iter"],
        tol=cfg["cg_tol"],
    ),
)
benchmark_rows.append(row)

inverse_hvp_results["lissa_minibatch"], row = benchmark_inverse_hvp(
    "lissa_minibatch",
    lambda: inverse_hvp_lissa(
        top_model,
        train_x,
        train_y,
        test_grad,
        cfg["weight_decay"],
        damping=0.0,
        batch_size=cfg["lissa_batch_size"],
        recursion_depth=cfg["lissa_depth"],
        scale=cfg["lissa_scale"],
        num_samples=cfg["lissa_num_samples"],
    ),
)
benchmark_rows.append(row)
print_benchmark_table(benchmark_rows)

influence_results = {}
for method_name, inverse_hvp in inverse_hvp_results.items():
    up_scores = upweight_effect(train_grads, inverse_hvp)
    loo_scores = leave_one_out_approx(up_scores, train_x.shape[0])
    helpful, harmful = helpful_harmful_indices(loo_scores, cfg["top_k"])
    influence_results[method_name] = {
        "inverse_hvp": inverse_hvp,
        "up_scores": up_scores,
        "loo_scores": loo_scores,
        "helpful": helpful,
        "harmful": harmful,
    }

selected_method = cfg["selected_method"]
selected = influence_results[selected_method]
up_scores = selected["up_scores"]
loo_scores = selected["loo_scores"]
helpful = selected["helpful"]
harmful = selected["harmful"]
print(f"\nSelected method for visualization: {selected_method}")

## 6. Visualize Helpful and Harmful Training Images

Helpful examples are points whose removal is predicted to increase the selected test loss. Harmful examples are points whose removal is predicted to decrease the selected test loss.

In [ ]:
def show_influence_grid(indices: Iterable[int], title: str) -> None:
    indices = [int(i) for i in (indices.detach().cpu().tolist() if hasattr(indices, "detach") else indices)]
    count = len(indices)
    cols = min(count, 5)
    rows = math.ceil(count / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes[count:]:
        ax.axis("off")

    for rank, (ax, idx) in enumerate(zip(axes, indices), start=1):
        img = Image.open(train_paths[idx]).convert("RGB")
        label_name = cfg["classes"][int(train_y_cpu[idx].item())]
        loo = float(loo_scores[idx].detach().cpu().item())
        up = float(up_scores[idx].detach().cpu().item())
        ax.imshow(img)
        ax.set_title(f"#{rank} idx={idx} {label_name}\nLOO={loo:.2e}\nup={up:.2e}", fontsize=9)
        ax.axis("off")

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


show_influence_grid(helpful, f"Most helpful training images ({selected_method})")
show_influence_grid(harmful, f"Most harmful training images ({selected_method})")

bar_indices = torch.cat([helpful, harmful]).detach().cpu().tolist()
bar_values = [float(loo_scores[int(i)].detach().cpu().item()) for i in bar_indices]
bar_labels = [f"{int(i)}\n{cfg['classes'][int(train_y_cpu[int(i)].item())]}" for i in bar_indices]
colors = ["tab:green"] * len(helpful) + ["tab:red"] * len(harmful)

plt.figure(figsize=(max(8, 0.8 * len(bar_indices)), 4))
plt.bar(range(len(bar_indices)), bar_values, color=colors)
plt.axhline(0.0, color="black", linewidth=1)
plt.xticks(range(len(bar_indices)), bar_labels, rotation=0)
plt.ylabel("predicted LOO test-loss delta")
plt.title(f"Helpful vs harmful influence scores ({selected_method})")
plt.tight_layout()
plt.show()

## 7. Optional Leave-One-Out Sanity Check

This retrains only the top logistic classifier on cached Inception features after removing selected training points.

In [ ]:
def retrain_without_one(remove_idx: int) -> float:
    mask = torch.ones(train_x.shape[0], dtype=torch.bool, device=device)
    mask[remove_idx] = False
    model = train_top_model(train_x[mask], train_y[mask], cfg)
    with torch.no_grad():
        logit = model(test_feature)
        loss = F.binary_cross_entropy_with_logits(logit, test_label, reduction="sum")
    return float(loss.item())


if cfg["sanity_checks"] > 0:
    print("Leave-one-out sanity check")
    print(" train_idx | predicted_delta | actual_delta")
    print("-----------+-----------------+-------------")
    candidates = torch.cat([helpful, harmful])[: cfg["sanity_checks"]]
    for idx_tensor in candidates:
        idx = int(idx_tensor)
        actual_loss = retrain_without_one(idx)
        actual_delta = actual_loss - test_loss
        pred_delta = float(loo_scores[idx].detach().cpu().item())
        print(f" {idx:>9} | {pred_delta:>15.8f} | {actual_delta:>11.8f}")

## 8. Commented Output Saving

Saving is intentionally commented out by default. Uncomment these lines if you want to persist features, influence arrays, or benchmark metadata.

In [ ]:
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# np.savez(
#     OUTPUT_DIR / f"dogfish_influence_{CONFIG_NAME}.npz",
#     benchmark=np.array(benchmark_rows, dtype=object),
#     selected_method=selected_method,
#     upweight_effect=up_scores.detach().cpu().numpy(),
#     leave_one_out_delta=loo_scores.detach().cpu().numpy(),
#     helpful_indices=helpful.detach().cpu().numpy(),
#     harmful_indices=harmful.detach().cpu().numpy(),
#     train_paths=np.array([str(p) for p in train_paths], dtype=object),
#     test_paths=np.array([str(p) for p in test_paths], dtype=object),
#     test_index=test_index,
#     config_name=CONFIG_NAME,
# )
# torch.save(
#     {
#         "top_model_state_dict": top_model.state_dict(),
#         "config": cfg,
#         "selected_method": selected_method,
#     },
#     OUTPUT_DIR / f"dogfish_top_model_{CONFIG_NAME}.pt",
# )